# Решение задачи интеллектуального анализа данных (часть 1)

Проект «Подбор домов Красноярска по анкете». Одна запись — оценка адреса по анкете. Целевой ответ — «Добавили бы вы этот дом в список для просмотра?».

Каталог содержит реальные адреса и расстояния по прямой. Анкеты и ответы сгенерированы для учебного исследования. Запускайте все ячейки. CSV берутся из папки **data** рядом с ноутбуком.

[Данные каталога © OpenStreetMap contributors, ODbL](https://www.openstreetmap.org/copyright).

## Загрузка выборки

80% анкет используются для обучения и кросс-валидации. Оставшиеся 20% анкет целиком отложены для финального теста.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

project = Path.cwd()
if (project / 'house_matching' / 'prepare_data.py').is_file():
    project = project / 'house_matching'
sys.path.insert(0, str(project))

from prepare_data import (
    DISTANCES, FEATURES, HOUSES_PER_SURVEY, LIMITS, PARKING, PLACES,
    PRIORITIES, SEED, SURVEY_COUNT, TARGET,
)

numeric = DISTANCES + LIMITS + PRIORITIES

houses = pd.read_csv(project / 'data' / 'houses.csv', dtype={'floors': 'Int64'})
surveys = pd.read_csv(project / 'data' / 'surveys.csv')
answers = pd.read_csv(project / 'data' / 'answers.csv')
pairs = answers.merge(houses, on='house_id', validate='many_to_one')
pairs = pairs.merge(surveys, on='survey_id', validate='many_to_one')

assert len(FEATURES) == 23 and len(surveys) == SURVEY_COUNT
assert len(pairs) == SURVEY_COUNT * HOUSES_PER_SURVEY
assert not pairs.duplicated(['survey_id', 'house_id']).any()
assert pairs.groupby('survey_id').size().eq(HOUSES_PER_SURVEY).all()
assert pairs[TARGET].isin([0, 1]).all()
fits_district = [district in district_list.split('|')
                 for district, district_list in zip(pairs['district'], pairs['districts'])]
assert all(fits_district)

train_ids, test_ids = train_test_split(surveys['survey_id'], test_size=.2, random_state=SEED)
assert set(train_ids).isdisjoint(test_ids)
eda = pairs.loc[pairs['survey_id'].isin(train_ids)].copy()
print(f'{len(houses):,} адресов · {len(surveys)} анкет · {len(pairs):,} ответов'.replace(',', ' '))
print(f'Для анализа отклика {len(train_ids)} анкет, для финального теста {len(test_ids)}.')

22 065 адресов · 600 анкет · 18 000 ответов
Для анализа отклика 480 анкет, для финального теста 120.


### Базовый подбор

Сначала оставляем выбранные районы. Вклад расстояния d при желаемом T равен T / (T + d).
У объекта на желаемом расстоянии вклад равен 0,5. Более близкие объекты получают больший балл.
Для парковки вклады составляют 1, 0,5 и 0 для трёх возрастающих интервалов.

Балл дома — средний вклад с учётом важности. Дома ранжируются по баллу, при равенстве — по номеру.
Для анкеты без приоритетов сортировка идёт по расстоянию до остановки.

In [2]:
def baseline_score(frame):
    distance = frame[DISTANCES].to_numpy(dtype=float)
    desired = frame[LIMITS].to_numpy(dtype=float)
    distance_credit = desired / (desired + distance)
    parking_credit = frame['parking'].map(dict(zip(PARKING, [1, .5, 0]))).to_numpy(dtype=float)
    weights = frame[PRIORITIES].to_numpy(dtype=float)
    total = weights.sum(axis=1)
    earned = (np.column_stack([distance_credit, parking_credit]) * weights).sum(axis=1)
    by_stop = 1 / (1 + frame['stop_m'].to_numpy(dtype=float) / 300)
    return np.divide(earned, total, out=by_stop, where=total > 0)


def candidates(profile):
    districts = profile['districts'] or houses['district'].unique()
    unknown = set(districts) - set(houses['district'])
    if unknown:
        raise ValueError('Неизвестные районы: ' + ', '.join(sorted(unknown)))

    result = houses.loc[houses['district'].isin(districts)].copy()
    for place, limit in zip(PLACES, LIMITS):
        result[limit] = profile['limits_m'].get(place, surveys[limit].median())
    for place, priority in zip([*PLACES, 'parking'], PRIORITIES):
        result[priority] = profile['importance'].get(place, 0)
    if not (np.isfinite(result[LIMITS]).all().all() and result[LIMITS].gt(0).all().all()):
        raise ValueError('Укажите положительные расстояния в метрах.')
    if not result[PRIORITIES].isin(range(4)).all().all():
        raise ValueError('Укажите важность от 0 до 3.')
    return result


family_profile = {
    'districts': ['Октябрьский', 'Советский', 'Центральный'],
    'limits_m': {'stop': 350, 'school': 450, 'kindergarten': 450, 'park': 650, 'mall': 1500},
    'importance': {'stop': 2, 'school': 3, 'kindergarten': 3, 'park': 2, 'mall': 1, 'parking': 1},
}

## Подготовка признаков

Проверяем, поднимает ли модель в первые пять позиций дома с положительным ответом.
Прогноз строится по 23 исходным признакам и их сопоставлениям. Для каждого объекта рассчитывается
нормированное расхождение между желаемым и фактическим расстоянием с учётом важности.
Отдельно учитываются совместное выполнение пожеланий по школе и детсаду, а также по аптеке и больнице.

Масштабирование применяется здесь. Медианы, параметры StandardScaler и категории OneHotEncoder
оцениваются внутри обучающей части каждого блока. Для крупных значений по IQR сохраняем измеренные
расстояния. Отрицательные расстояния, неположительные пороги и баллы вне шкалы исключаются как ошибки.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, ndcg_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

invalid_rows = (
    eda[DISTANCES].lt(0).any(axis=1) | eda[LIMITS].le(0).any(axis=1)
    | ((~eda[PRIORITIES].isin(range(4))) & eda[PRIORITIES].notna()).any(axis=1)
)
train_rows = eda.loc[~invalid_rows].copy()
test_rows = pairs.loc[pairs['survey_id'].isin(test_ids)].copy()
print('Некорректных пар исключено из обучения:', invalid_rows.sum())
print(f'Обучение: {len(train_rows)} пар. Тест: {len(test_rows)} пар.')

match_columns = [f'match_{name}' for name in (*PLACES, 'parking')]
joint_columns = ['education_within', 'health_within']


def model_features(frame):
    result = frame[FEATURES].copy()
    for place, distance, limit, priority in zip(PLACES, DISTANCES, LIMITS, PRIORITIES):
        result[f'match_{place}'] = (frame[limit] - frame[distance]) / (frame[limit] + frame[distance])
        result[f'match_{place}'] *= frame[priority]
    result['match_parking'] = frame['parking'].map(dict(zip(PARKING, [1, .5, 0]))) * frame['priority_parking']
    for column, first, second in [
        ('education_within', 'school', 'kindergarten'),
        ('health_within', 'pharmacy', 'hospital'),
    ]:
        result[column] = (
            frame[f'{first}_m'].le(frame[f'want_{first}_m'])
            & frame[f'{second}_m'].le(frame[f'want_{second}_m'])
            & frame[f'priority_{first}'].ge(2) & frame[f'priority_{second}'].ge(2)
        ).astype(int)
    return result


keys = ['survey_id', 'house_id']
X_train = train_rows[keys].join(model_features(train_rows))
X_test = test_rows[keys].join(model_features(test_rows))
y_train, y_test = train_rows[TARGET], test_rows[TARGET]

transform = ColumnTransformer([
    ('numeric', Pipeline([('fill', SimpleImputer(strategy='median')), ('scale', StandardScaler())]),
     numeric + match_columns + joint_columns),
    ('parking', Pipeline([('fill', SimpleImputer(strategy='most_frequent')),
                          ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
     ['parking']),
])

cv = GroupKFold(n_splits=5)
for train_index, valid_index in cv.split(X_train, y_train, groups=X_train['survey_id']):
    assert set(X_train.iloc[train_index]['survey_id']).isdisjoint(X_train.iloc[valid_index]['survey_id'])
assert len(y_train) == 14_400 and len(y_test) == 3_600

Некорректных пар исключено из обучения: 0
Обучение: 14400 пар. Тест: 3600 пар.


## Метрики и подбор гиперпараметров

- **NDCG@5** оценивает позиции подходящих домов. Первые позиции имеют больший вес.
- **Precision@5** — доля ответов «да» среди пяти предложенных адресов.
- **Recall@5** — доля всех положительно оценённых адресов анкеты, попавших в пятёрку.
- **AP** — средняя точность для всех пар при разных порогах. Она учитывает ранжирование пар по скору.

Первые три метрики усредняются по анкетам. Для анкеты без ответов «да» Recall@5 и NDCG@5 равны нулю.
Модель и параметры выбираем по NDCG@5 на пяти блоках обучения.
Равные оценки упорядочиваются по номеру записи в каталоге.

In [4]:
def ranking_arrays(X, y, scores):
    # Each survey has exactly 30 assessed addresses. Sorting makes tie-breaking reproducible.
    order = np.lexsort((X['house_id'].to_numpy(), X['survey_id'].to_numpy()))
    assert X.groupby('survey_id').size().eq(HOUSES_PER_SURVEY).all()
    observed = np.asarray(y)[order].reshape(-1, HOUSES_PER_SURVEY)
    scores = np.asarray(scores)[order].reshape(-1, HOUSES_PER_SURVEY)
    ranks = np.argsort(-scores, axis=1, kind='stable')
    relevance = np.take_along_axis(observed, ranks, axis=1)
    return relevance


def mean_ndcg_at_five(relevance):
    # Relevance is already ordered from the first suggestion to the last.
    rank_scores = np.broadcast_to(np.arange(relevance.shape[1], 0, -1), relevance.shape)
    return ndcg_score(relevance, rank_scores, k=5)


def ranking_metrics(X, y, scores):
    relevance = ranking_arrays(X, y, scores)
    hits = relevance[:, :5].sum(axis=1)
    total = relevance.sum(axis=1)
    recall = np.divide(hits, total, out=np.zeros(len(total)), where=total > 0)
    return {'NDCG@5': mean_ndcg_at_five(relevance), 'Precision@5': (hits / 5).mean(),
            'Recall@5': recall.mean(), 'AP': average_precision_score(y, scores)}


def cv_ndcg(estimator, X, y):
    scores = estimator.predict_proba(X)[:, 1]
    return mean_ndcg_at_five(ranking_arrays(X, y, scores))


models = [
    ('Логистическая регрессия', LogisticRegression(max_iter=1000, random_state=SEED),
     {'model__C': [.1, 1, 10], 'model__class_weight': [None, 'balanced']}),
    ('KNN', KNeighborsClassifier(),
     {'model__n_neighbors': [7, 21, 45], 'model__weights': ['uniform', 'distance']}),
    ('Случайный лес', RandomForestClassifier(random_state=SEED, min_samples_leaf=6, n_jobs=2),
     {'model__n_estimators': [60, 120], 'model__max_depth': [7, 12]}),
]

searches = {}
for name, estimator, parameters in models:
    search = GridSearchCV(
        Pipeline([('prepare', transform), ('model', estimator)]), parameters,
        cv=cv, scoring=cv_ndcg, n_jobs=1, error_score='raise',
    )
    search.fit(X_train, y_train, groups=X_train['survey_id'])
    searches[name] = search
    best_parameters = {key.removeprefix('model__'): value for key, value in search.best_params_.items()}
    print(f'{name}: {len(search.cv_results_["params"])} комбинаций, '
          f'NDCG@5 на CV = {search.best_score_:.3f}, параметры {best_parameters}')

Логистическая регрессия: 6 комбинаций, NDCG@5 на CV = 0.811, параметры {'C': 1, 'class_weight': None}
KNN: 6 комбинаций, NDCG@5 на CV = 0.774, параметры {'n_neighbors': 45, 'weights': 'uniform'}
Случайный лес: 4 комбинаций, NDCG@5 на CV = 0.806, параметры {'max_depth': 12, 'n_estimators': 60}


## Финальный тест на новых анкетах

В тест попали все 30 оценок каждой из 120 отложенных анкет.
Простой метод оценивается на тех же парах и по тем же метрикам.

In [5]:
base_name = 'Взвешенное правило'
base_cv = mean_ndcg_at_five(ranking_arrays(X_train, y_train, baseline_score(train_rows)))
best_ml = max(searches, key=lambda name: searches[name].best_score_)
method = best_ml if searches[best_ml].best_score_ > base_cv else base_name

results = [{'Метод': base_name, 'NDCG@5 (CV)': base_cv,
             **ranking_metrics(X_test, y_test, baseline_score(test_rows)) }]
for name, search in searches.items():
    metrics = ranking_metrics(X_test, y_test, search.predict_proba(X_test)[:, 1])
    results.append({'Метод': name, 'NDCG@5 (CV)': search.best_score_, **metrics})

comparison = pd.DataFrame(results).set_index('Метод')
display(comparison.round(3))
delta = comparison.loc[best_ml, 'NDCG@5'] - comparison.loc[base_name, 'NDCG@5']
print('Тестовых анкет без положительных ответов:', y_test.groupby(X_test['survey_id']).sum().eq(0).sum())
display(Markdown(
    f'По CV выбран метод **«{method}»**. Среди обучаемых методов лучший на CV — **{best_ml}**. '
    f'NDCG@5 на тесте **{comparison.loc[best_ml, "NDCG@5"]:.3f}** у модели и '
    f'**{comparison.loc[base_name, "NDCG@5"]:.3f}** у правила. Разница **{delta:+.3f}**. '
    'Результат относится к учебным ответам и кандидатам, оценённым в каждой анкете.'
))

Тестовых анкет без положительных ответов: 0


,NDCG@5 (CV),NDCG@5,Precision@5,Recall@5,AP
Метод,,,,,
Взвешенное правило,0.808,0.782,0.755,0.305,0.748
Логистическая регрессия,0.811,0.786,0.765,0.307,0.758
KNN,0.774,0.757,0.742,0.303,0.720
Случайный лес,0.806,0.760,0.742,0.301,0.743


По CV выбран метод **«Логистическая регрессия»**. Среди обучаемых методов лучший на CV — **Логистическая регрессия**. NDCG@5 на тесте **0.786** у модели и **0.782** у правила. Разница **+0.004**. Результат относится к учебным ответам и кандидатам, оценённым в каждой анкете.

## Выдача для двух анкет

Пример 1 отдаёт приоритет школе и детсаду. Пример 2 отдаёт приоритет аптеке и больнице.
Районы одинаковы, поэтому можно увидеть, как список меняется из-за пожеланий.
Чтобы выполнить собственный подбор, измените районы, расстояния и важность в словаре.
Балл используется для сортировки адресов. Для конкретной квартиры дополнительно проверяются объявления,
цена, состояние дома и маршруты.

In [6]:
health_profile = {
    'districts': family_profile['districts'],
    'limits_m': {'stop': 300, 'pharmacy': 400, 'hospital': 650, 'park': 600},
    'importance': {'stop': 2, 'pharmacy': 3, 'hospital': 3, 'park': 1},
}


def recommend(profile):
    pool = candidates(profile)
    pool['rule_score'] = baseline_score(pool)
    effective_method = method if pool[PRIORITIES].to_numpy().any() else base_name
    if effective_method == base_name:
        pool['score'] = pool['rule_score']
    else:
        pool['score'] = searches[effective_method].predict_proba(model_features(pool))[:, 1]
    best = pool.sort_values(['score', 'house_id'], ascending=[False, True]).head(5).copy()
    required = sorted((place for place in PLACES if profile['importance'].get(place, 0) > 0),
                      key=lambda place: -profile['importance'][place])

    for place in required:
        name = PLACES[place][1]
        best[f'{name}, м'] = [
            f'{distance:.0f} / {goal:.0f} {"✓" if distance <= goal else "↑"}'
            for distance, goal in zip(best[f'{place}_m'], best[f'want_{place}_m'])
        ]
    columns = ['address', 'district', 'floors', 'score'] + [PLACES[p][1] + ', м' for p in required]
    if profile['importance'].get('parking', 0) > 0:
        columns.append('parking')
    shown = best[columns].rename(columns={
        'address': 'Адрес', 'district': 'Район', 'floors': 'Этажей', 'score': 'Балл', 'parking': 'Парковка',
    })
    shown.index = range(1, len(shown) + 1)
    return shown.round(3)


for example, questionnaire in [('Школа и детсад', family_profile), ('Аптека и больница', health_profile)]:
    recommendations = recommend(questionnaire)
    assert len(recommendations) == 5 and recommendations['Район'].isin(questionnaire['districts']).all()
    display(Markdown(f'**{example}**'))
    display(recommendations)

**Школа и детсад**

,Адрес,Район,Этажей,Балл,"Школа, м","Детсад, м","Остановка, м","Парк, м","ТЦ, м",Парковка
1,улица Марковского 56,Центральный,7,0.902,38 / 450 ✓,62 / 450 ✓,75 / 350 ✓,97 / 650 ✓,647 / 1500 ✓,До 150 м
2,улица Марковского 45/2,Центральный,1,0.898,57 / 450 ✓,54 / 450 ✓,66 / 350 ✓,127 / 650 ✓,580 / 1500 ✓,До 150 м
3,улица Марковского 49,Центральный,9,0.893,98 / 450 ✓,74 / 450 ✓,62 / 350 ✓,65 / 650 ✓,524 / 1500 ✓,До 150 м
4,улица Вейнбаума 38,Центральный,9,0.889,66 / 450 ✓,126 / 450 ✓,79 / 350 ✓,36 / 650 ✓,657 / 1500 ✓,До 150 м
5,улица Марковского 45,Центральный,5,0.889,69 / 450 ✓,61 / 450 ✓,81 / 350 ✓,143 / 650 ✓,583 / 1500 ✓,До 150 м


**Аптека и больница**

,Адрес,Район,Этажей,Балл,"Аптека, м","Больница, м","Остановка, м","Парк, м"
1,улица Ады Лебедевой 91,Центральный,5,0.876,23 / 400 ✓,55 / 650 ✓,11 / 300 ✓,129 / 600 ✓
2,Аэровокзальная улица 3,Советский,4,0.861,6 / 400 ✓,125 / 650 ✓,22 / 300 ✓,182 / 600 ✓
3,улица Ады Лебедевой 93,Центральный,2,0.861,22 / 400 ✓,88 / 650 ✓,41 / 300 ✓,127 / 600 ✓
4,улица Весны 11,Советский,10,0.856,43 / 400 ✓,13 / 650 ✓,62 / 300 ✓,202 / 600 ✓
5,улица Карла Маркса 47,Центральный,9,0.851,10 / 400 ✓,164 / 650 ✓,25 / 300 ✓,172 / 600 ✓


✓ — расстояние в пределах пожелания, ↑ — превышение. В ячейке указаны фактическое и желаемое расстояния.

Следующий этап проверки — повторить эксперимент с независимыми ответами жителей на карточки домов.
Для УК полезны агрегированные пожелания и доли доступных адресов по районам, как в дашборде.